In [ ]:
TRAIN_CSV_PATH = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/train.csv"
TEST_CSV_PATH = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/test.csv"
SAMPLE_SUBMISSION_CSV_PATH = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/sample_submission.csv"
SUBMISSION_CSV_PATH = "submission.csv"
IMG_DIR = "/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/images"

In [ ]:
import pandas as pd

df = pd.read_csv(TRAIN_CSV_PATH)

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nSample rows:")
display(df.head())

In [ ]:
label_cols = df.columns[1:]

row_sums = df[label_cols].sum(axis=1)

print("Unique values in row sum:", row_sums.unique())

print("\nCounts of invalid rows:")
print((row_sums != 1).sum())

In [ ]:
import matplotlib.pyplot as plt

class_counts = df[label_cols].sum().sort_values(ascending=False)

print(class_counts)

plt.figure(figsize=(12,6))
class_counts.plot(kind='bar')
plt.title("Class Distribution")
plt.xticks(rotation=90)
plt.show()

In [ ]:
print("Most frequent class count:", class_counts.max())
print("Least frequent class count:", class_counts.min())

imbalance_ratio = class_counts.max() / class_counts.min()
print("Imbalance ratio:", imbalance_ratio)

In [ ]:
df['label'] = df[label_cols].values.argmax(axis=1)

print(df[['id', 'label']].head())

In [ ]:
df['label'].value_counts().sort_index()

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt

IMG_DIR = '/kaggle/input/competitions/26-t-1-dl-gen-ainppe-1/images'

def show_samples(class_id, n=5):
    samples = df[df['label'] == class_id].sample(n)
    
    plt.figure(figsize=(15,3))
    
    for i, (_, row) in enumerate(samples.iterrows()):
        img_path = os.path.join(IMG_DIR, row['id'])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(1, n, i+1)
        plt.imshow(img)
        plt.title(f"Class {class_id}")
        plt.axis('off')
    
    plt.show()

# Try a few classes
for i in range(20):
    show_samples(i)

In [ ]:
from tqdm import tqdm

sizes = []

for img_name in tqdm(df['id'].sample(1000)):  # sample for speed
    img_path = os.path.join(IMG_DIR, img_name)
    img = cv2.imread(img_path)
    sizes.append(img.shape[:2])  # (H, W)

sizes_df = pd.DataFrame(sizes, columns=['height', 'width'])

print(sizes_df.describe())

In [ ]:
no_finding_count = df['No Finding'].sum()
total = len(df)

print("No Finding %:", no_finding_count / total)

In [ ]:
rare_classes = class_counts[class_counts < 100]

print("Rare classes:")
print(rare_classes)

In [ ]:
# =========================
# IMPORTS
# =========================

import os
import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.models import DenseNet121_Weights

from sklearn.model_selection import train_test_split
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
# =========================
# SETTINGS
# =========================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_CLASSES = 20
BATCH_SIZE = 32
EPOCHS = 200

EARLY_STOP_DELTA = 1e-3
PATIENCE = 5

SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok = True)

In [ ]:
# =========================
# LABELS
# =========================

label_cols = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion",
    "Emphysema", "Fibrosis", "Hernia", "Infiltration", "Mass", "Nodule",
    "Pleural_Thickening", "Pneumonia", "Pneumothorax", "Pneumoperitoneum",
    "Pneumomediastinum", "Subcutaneous Emphysema", "Tortuous Aorta",
    "Calcification of the Aorta", "No Finding"
]

NO_FINDING_IDX = 19

In [ ]:
# =========================
# LOAD DATA
# =========================

df = pd.read_csv(TRAIN_CSV_PATH)

train_df, val_df = train_test_split(
    df,
    test_size = 0.1,
    random_state = 42
)

In [ ]:
# =========================
# LIMIT NO FINDING
# =========================

mask     = train_df[label_cols].iloc[:, NO_FINDING_IDX] == 1
df_no    = train_df[mask].sample(5000, random_state = 42)
df_other = train_df[~mask]

train_df = pd.concat([df_no, df_other]).reset_index(drop = True)

In [ ]:
# =========================
# AUGMENTATIONS
# =========================

train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p = 0.5),
    A.Affine(scale = (0.95, 1.05), translate_percent = 0.05, rotate = (-10, 10), p = 0.5),
    A.Normalize(),
    ToTensorV2()
])

valid_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
# =========================
# DATASET
# =========================

class XrayDataset(Dataset):

    def __init__(self, df, tfms):
        self.df   = df.reset_index(drop = True)
        self.tfms = tfms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = os.path.join(IMG_DIR, row["id"])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.tfms(image = image)["image"]

        label = torch.tensor(row[label_cols].values.astype(np.float32))

        return image, label

In [ ]:
# =========================
# DATALOADERS
# =========================

train_loader = DataLoader(
    XrayDataset(train_df, train_tfms),
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 2
)

val_loader = DataLoader(
    XrayDataset(val_df, valid_tfms),
    batch_size = BATCH_SIZE,
    shuffle = False,
    num_workers = 2
)

In [ ]:
# =========================
# MODEL
# =========================

model = models.densenet121(
    weights = DenseNet121_Weights.IMAGENET1K_V1
)

for name, param in model.features.named_parameters():
    if "denseblock4" in name or "norm5" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

model.classifier = nn.Linear(
    model.classifier.in_features,
    NUM_CLASSES
)

model = model.to(DEVICE)

In [ ]:
# =========================
# LOSS + OPTIMIZER
# =========================

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW([
    {"params": model.features.parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3}
])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max = EPOCHS
)

In [ ]:
# =========================
# TRAIN
# =========================

scaler = torch.amp.GradScaler("cuda")

best_loss        = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for imgs, labels in tqdm(train_loader):

        imgs   = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}: Loss = {avg_loss:.4f}")

In [ ]:
    # =========================
    # SAVE EVERY EPOCH
    # =========================

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss": avg_loss
    }, f"{SAVE_DIR}/epoch_{epoch + 1}.pth")

    # =========================
    # SAVE BEST MODEL
    # =========================

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pth")
        print("Saved Best Model")

    # =========================
    # EARLY STOP
    # =========================

    if best_loss - avg_loss < EARLY_STOP_DELTA:
        patience_counter += 1
    else:
        patience_counter = 0

    if patience_counter >= PATIENCE:
        print("Early stopping triggered")
        break

In [ ]:
# =========================
# LOAD BEST MODEL
# =========================

model.load_state_dict(
    torch.load(f"{SAVE_DIR}/best_model.pth")
)

model.eval()

In [ ]:
# =========================
# TEST DATASET
# =========================

test_df = pd.read_csv(TEST_CSV_PATH)

class TestDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(drop = True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join(IMG_DIR, self.df.iloc[idx]["id"])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = valid_tfms(image = image)["image"]

        return image

In [ ]:
# =========================
# SUBMISSION
# =========================

submission = pd.DataFrame(
    test_preds,
    columns = label_cols
)

submission.insert(0, "id", test_df["id"])

submission.to_csv("submission.csv", index = False)

print(f"{ "=" * 20}DONE{ "=" * 20}")